# Total features + RFECV + LightGBM check notebook

Notebook này dùng để kiểm tra pipeline:

- `research_total_features_rfecv_lgbm.py`
- cache `initial_non_bb_candidates`
- scaling `rolling_zscore`
- feature selection `RFECV`
- model `LightGBM`


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT = Path(r"D:\SCJ999\srateries\RandomForest\RF_MLFlow")
CACHE_FAMILY = "initial_non_bb_candidates"
SCRIPT = PROJECT / "research_total_features_rfecv_lgbm.py"
assert SCRIPT.exists(), SCRIPT
%cd $PROJECT


D:\SCJ999\srateries\RandomForest\RF_MLFlow


## 1. Kiểm tra dependency

In [2]:
import lightgbm as lgb
import sklearn
print("lightgbm", lgb.__version__)
print("sklearn", sklearn.__version__)


lightgbm 4.6.0
sklearn 1.5.2


## 2. Kiểm tra cache feature columns

Nếu overlay feature count bằng 0 thì cache chưa rebuild sau khi thêm overlay indicators.

In [3]:
cols_path = PROJECT / "cache" / CACHE_FAMILY / "feature_columns.json"
cols = json.loads(cols_path.read_text(encoding="utf-8"))

summary = {
    "cache_family": CACHE_FAMILY,
    "total_features": len(cols),
    "overlay_features": sum("_ov_" in c for c in cols),
    "featuretools_features": sum(c.startswith("ft_") for c in cols),
    "time_features": sum(c in {"tod_sin", "tod_cos", "dow_sin", "dow_cos"} for c in cols),
    "signal_features": sum(c.startswith("sig_") or c == "buy_signal" for c in cols),
}
summary


{'cache_family': 'initial_non_bb_candidates',
 'total_features': 1404,
 'overlay_features': 1224,
 'featuretools_features': 0,
 'time_features': 4,
 'signal_features': 4}

In [4]:
pd.Series(cols).to_frame("feature").assign(
    is_overlay=lambda d: d["feature"].str.contains("_ov_", regex=False),
    family=lambda d: np.select(
        [
            d["feature"].str.startswith(("ema", "sma")) & d["feature"].str.contains("_ov_", regex=False),
            d["feature"].str.startswith("bb"),
            d["feature"].str.startswith(("keltner", "donchian")),
            d["feature"].str.startswith("vwap"),
            d["feature"].str.startswith(("ichimoku", "parabolic_sar", "supertrend")),
            d["feature"].str.startswith("h4_"),
            d["feature"].str.startswith("h1_"),
            d["feature"].str.startswith("m15_"),
            d["feature"].str.startswith("m5_"),
        ],
        ["ma_overlay", "bb_overlay", "channel_overlay", "vwap_overlay", "trendline_overlay", "h4", "h1", "m15", "m5"],
        default="other",
    ),
).groupby("family").size().sort_values(ascending=False)


family
ma_overlay           408
bb_overlay           306
channel_overlay      306
trendline_overlay    204
other                120
h1                    15
h4                    15
m15                   15
m5                    15
dtype: int64

## 3. Kiểm tra parquet sample

Cell này chỉ đọc metadata/cột của năm 2018 để xác nhận cache thực tế khớp `feature_columns.json`.

In [5]:
sample_path = PROJECT / "cache" / CACHE_FAMILY / "candidates_2018.parquet"
sample = pd.read_parquet(sample_path)
meta = {"dates", "candle_index", "entry_index", "entry_price", "year", "label"}
sample_features = [c for c in sample.columns if c not in meta]
{
    "rows_2018": len(sample),
    "parquet_cols": len(sample.columns),
    "parquet_features": len(sample_features),
    "overlay_features": sum("_ov_" in c for c in sample_features),
    "label_rate": float(sample["label"].mean()) if "label" in sample else None,
}


{'rows_2018': 112082,
 'parquet_cols': 1410,
 'parquet_features': 1404,
 'overlay_features': 1224,
 'label_rate': 0.47217215966881393}

## 4. Optional: rebuild cache overlay

Chỉ chạy cell này nếu cache chưa có overlay hoặc bạn muốn rebuild lại từ đầu. Có thể tốn RAM/thời gian.

In [8]:
# from research_random50_initial_features import build_cache
# build_cache(True)


## 5. Smoke test pipeline

Chạy rất nhỏ để xác nhận script không lỗi. Không dùng kết quả này để đánh giá model.

In [9]:
import time
RUN_ID = int(time.time())
RUN_ID


1788877917

In [11]:
!python research_total_features_rfecv_lgbm.py ^
  --run-id {RUN_ID} ^
  --cache-family initial_non_bb_candidates ^
  --rfecv-rows 1500 ^
  --max-train-rows 3000 ^
  --min-features-to-select 20 ^
  --rfecv-step 300 ^
  --cv-splits 2 ^
  --n-estimators-rfecv 20 ^
  --n-estimators-final 40 ^
  --n-jobs 2 ^
  --log-diagnostics ^
  --shap-rows 0


IndentationError: unexpected indent (2831647701.py, line 2)

## 6. RAM-safe run

Chạy bản nghiên cứu nhỏ vừa đủ. Dự kiến lâu hơn smoke test nhiều lần.

In [ ]:
# RUN_ID = int(time.time())
# !python research_total_features_rfecv_lgbm.py ^
#   --run-id {RUN_ID} ^
#   --cache-family initial_non_bb_candidates ^
#   --rfecv-rows 20000 ^
#   --max-train-rows 60000 ^
#   --min-features-to-select 40 ^
#   --rfecv-step 100 ^
#   --cv-splits 3 ^
#   --n-estimators-rfecv 150 ^
#   --n-estimators-final 700 ^
#   --n-jobs 2 ^
#   --enable-mlflow ^
#   --log-diagnostics ^
#   --shap-rows 0


## 7. Đọc kết quả của RUN_ID

In [12]:
out_dir = PROJECT / "outputs" / f"total_features_rfecv_lgbm_{RUN_ID}"
summary_path = out_dir / "summary.json"
metrics_path = out_dir / "metrics.json"
print(out_dir)
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary


D:\SCJ999\srateries\RandomForest\RF_MLFlow\outputs\total_features_rfecv_lgbm_1788877917


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\SCJ999\\srateries\\RandomForest\\RF_MLFlow\\outputs\\total_features_rfecv_lgbm_1788877917\\summary.json'

In [ ]:
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
metrics


## 8. RFECV ranking

In [ ]:
rank_path = out_dir / "rfecv_feature_ranking.csv"
rank = pd.read_csv(rank_path)
rank.head(30)


In [ ]:
rank.groupby(["selected", "ranking"]).size().reset_index(name="n").sort_values(["selected", "ranking"], ascending=[False, True]).head(30)


## 9. LightGBM feature importance

In [ ]:
imp_path = out_dir / "model" / "lgbm_feature_importance.csv"
imp = pd.read_csv(imp_path)
imp.head(50)


In [ ]:
imp.assign(
    family=lambda d: np.select(
        [
            d["feature"].str.contains("_ov_", regex=False),
            d["feature"].str.startswith("h4_"),
            d["feature"].str.startswith("h1_"),
            d["feature"].str.startswith("m15_"),
            d["feature"].str.startswith("m5_"),
            d["feature"].str.startswith("sig_"),
            d["feature"].str.startswith(("tod_", "dow_", "session_")),
        ],
        ["overlay", "h4", "h1", "m15", "m5", "signal", "time_session"],
        default="other",
    )
).groupby("family").agg(n=("feature", "count"), importance_sum=("importance", "sum")).sort_values("importance_sum", ascending=False)


## 10. Threshold/yearly/monthly diagnostics

In [ ]:
valid_curve = pd.read_csv(out_dir / "model" / "threshold_valid.csv")
test_curve = pd.read_csv(out_dir / "model" / "threshold_test.csv")
display(valid_curve.sort_values("winrate", ascending=False).head(20))
display(test_curve.sort_values("winrate", ascending=False).head(20))


In [ ]:
yearly = pd.read_csv(out_dir / "model" / "total_features_rfecv_lgbm_yearly_test.csv")
monthly = pd.read_csv(out_dir / "model" / "total_features_rfecv_lgbm_monthly_test.csv")
display(yearly)
display(monthly.head(30))


## 11. Hiển thị chart artifact

In [ ]:
from IPython.display import Image, display

chart_files = [
    out_dir / "model" / "lgbm_feature_importance_top40.png",
    out_dir / "model" / "total_features_rfecv_lgbm_roc_valid.png",
    out_dir / "model" / "total_features_rfecv_lgbm_roc_test.png",
    out_dir / "model" / "total_features_rfecv_lgbm_calibration_valid.png",
    out_dir / "model" / "total_features_rfecv_lgbm_calibration_test.png",
    out_dir / "model" / "total_features_rfecv_lgbm_prob_distribution_valid_test.png",
    out_dir / "model" / "total_features_rfecv_lgbm_yearly_test_selected_threshold.png",
    out_dir / "model" / "total_features_rfecv_lgbm_monthly_test_selected_threshold.png",
]
for p in chart_files:
    if p.exists():
        print(p.name)
        display(Image(filename=str(p)))


## 12. MLflow UI

Chạy cell dưới trong terminal/notebook nếu cần mở MLflow UI. Nếu UI đã chạy rồi thì không cần chạy lại.

In [ ]:
# !mlflow ui --backend-store-uri sqlite:///D:/SCJ999/mlflow.db --host 127.0.0.1 --port 5000
